In [ ]:
import threading
import time

import httpx
import uvicorn

from src.api.server import app
from src.pipeline.config import PipelineConfig

cfg = PipelineConfig()

server_thread = threading.Thread(
    target=uvicorn.run,
    kwargs={"app": app, "host": "0.0.0.0", "port": 8000, "log_level": "info"},
    daemon=True,
)
server_thread.start()
time.sleep(2)

base_url = cfg.api_base_url.rstrip("/")
endpoints = [
    f"{base_url}/market/diagnostic",
    f"{base_url}/signal/latest",
    f"{base_url}/news/recent",
    f"{base_url}/decisions/history?days=1",
]
for url in endpoints:
    resp = httpx.get(url, timeout=10.0)
    print(url, resp.status_code)


In [ ]:
import json

diagnostic = httpx.get(f"{base_url}/market/diagnostic", timeout=10.0).json()
signal = httpx.get(f"{base_url}/signal/latest", timeout=10.0).json()
news = httpx.get(f"{base_url}/news/recent", timeout=10.0).json()
decisions = httpx.get(
    f"{base_url}/decisions/history?days={cfg.decisions_cache_days}",
    timeout=10.0,
).json()

print("DIAGNOSTIC")
print(json.dumps(diagnostic, indent=2))
print("\nSIGNAL")
print(json.dumps(signal, indent=2))
print("\nNEWS")
print(json.dumps(news, indent=2))
print("\nDECISIONS")
print(json.dumps(decisions, indent=2))


In [ ]:
import asyncio

from src.agents.graph import get_last_state, run_agent_team

result = asyncio.run(run_agent_team(cfg))
state = get_last_state() or {}

print("=== BULLISH ANALYST ===")
print(state.get("bullish_report", ""))
print("\n=== BEARISH ANALYST ===")
print(state.get("bearish_report", ""))
print("\n=== TRADER ===")
print(state.get("trader_report", ""))
print("\n=== RISK MANAGER ===")
print(state.get("risk_report", ""))
print("\n=== MANAGER DECISION ===")
print(state.get("final_decision", result))


In [ ]:
import json

print(json.dumps(result, indent=2))


In [ ]:
import sqlalchemy

from src.environment.simulator import PaperTradingSimulator

engine = sqlalchemy.create_engine(cfg.db_url, future=True)
simulator = PaperTradingSimulator(cfg, engine)
execution = simulator.execute(result, current_price=85.4)
print(execution)
summary = simulator.get_summary()
print(summary)
